In [1]:
import json
import pandas as pd

file_path = '../data/raw_catalog.jsonl'
data = []

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)
print(f"Total assessments scraped: {len(df)}")
df.head()

Total assessments scraped: 377


,type,name,url,description,duration,test_type,adaptive_support,remote_support
0,1,ADO.NET (New),https://www.shl.com/products/product-catalog/v...,Multi-choice test that measures the knowledge ...,10,[Knowledge & Skills],No,Yes
1,1,Accounts Receivable Simulation (New),https://www.shl.com/products/product-catalog/v...,Simulated data entry test that measures the ab...,8,[Simulations],No,Yes
2,1,Accounts Payable Simulation (New),https://www.shl.com/products/product-catalog/v...,Simulated data entry test that measures the ab...,8,[Simulations],No,Yes
3,1,Accounts Receivable (New),https://www.shl.com/products/product-catalog/v...,Multiple-choice test that measures the knowled...,13,[Knowledge & Skills],No,Yes
4,1,Accounts Payable (New),https://www.shl.com/products/product-catalog/v...,Multiple-choice test that measures the knowled...,9,[Knowledge & Skills],No,Yes


In [2]:
import os
import json
import numpy as np
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

from sentence_transformers import SentenceTransformer
import chromadb
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

c:\Users\rahul\OneDrive\Desktop\Projects\assessment-recommendation\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class EmbeddingManager:
    """Handles embedding generation using HuggingFace SentenceTransformers."""
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        print(f"⚙️ Loading Embedding model: {self.model_name}...")
        self.model = SentenceTransformer(self.model_name)
        print(f"✅ Embedding model loaded. Dimension: {self.model.get_sentence_embedding_dimension()}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        print(f"🧠 Generating embeddings for {len(texts)} texts...")
        return self.model.encode(texts, show_progress_bar=False)

# --- TEST COMPONENT 1 ---
embedding_manager = EmbeddingManager()
test_vector = embedding_manager.generate_embeddings(["This is a test document."])
print(f"Test Vector Shape: {test_vector.shape}") # Should be (1, 384)

⚙️ Loading Embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2180.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded. Dimension: 384
🧠 Generating embeddings for 1 texts...
Test Vector Shape: (1, 384)


In [4]:
class VectorStore:
    """Manages the ChromaDB persistent vector store."""
    def __init__(self, collection_name: str = "shl_assessments", persist_directory: str = "../database/chroma_db"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        print(f"📁 Initializing ChromaDB client at {self.persist_directory}...")
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "SHL Assessment Catalog"}
        )
        print(f"✅ Vector store ready. Current documents in DB: {self.collection.count()}")

    def add_assessments(self, documents: List[str], metadatas: List[Dict[str, Any]], ids: List[str], embeddings: np.ndarray):
        print(f"💾 Upserting {len(documents)} assessments to the vector store...")
        self.collection.upsert(
            ids=ids,
            metadatas=metadatas,
            documents=documents,
            embeddings=embeddings.tolist()
        )
        print(f"✅ Successfully updated database. Total documents: {self.collection.count()}")

# --- TEST COMPONENT 2 ---
vector_store = VectorStore()

📁 Initializing ChromaDB client at ../database/chroma_db...
✅ Vector store ready. Current documents in DB: 377


In [5]:
print("--- Starting Data Ingestion ---")
documents, metadatas, ids = [], [], []

with open('../data/raw_catalog.jsonl', 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        item = json.loads(line)
        
        # Clean up types
        test_types_list = item.get("test_type", [])
        test_types_str = ", ".join(test_types_list) if isinstance(test_types_list, list) else ""
        
        try:
            duration_val = int(item.get("duration", 0))
        except (ValueError, TypeError):
            duration_val = 0

        # Build payloads
        rich_text = f"Assessment Name: {item['name']}\nCategory: {test_types_str}\nDescription: {item.get('description', '')}"
        
        metadata = {
            "name": str(item["name"]),
            "url": str(item["url"]),
            "test_types": test_types_str,
            "duration": duration_val,
            "remote_support": str(item.get("remote_support", "No")).strip(),
            "adaptive_support": str(item.get("adaptive_support", "No")).strip()
        }
        
        documents.append(rich_text)
        metadatas.append(metadata)
        ids.append(f"shl_{idx}")

# Execute pipeline
embeddings = embedding_manager.generate_embeddings(documents)
vector_store.add_assessments(documents, metadatas, ids, embeddings)

--- Starting Data Ingestion ---
🧠 Generating embeddings for 377 texts...
💾 Upserting 377 assessments to the vector store...
✅ Successfully updated database. Total documents: 377


In [6]:
class AssessmentRetriever:
    """Handles query-based semantic retrieval and metadata filtering."""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, filters: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        print(f"\n🔍 Retrieving top {top_k} for query: '{query}'")
        
        where_clause = None
        if filters:
            conditions = []
            if "remote_support" in filters and filters["remote_support"]:
                conditions.append({"remote_support": filters["remote_support"]})
            if "adaptive_support" in filters and filters["adaptive_support"]:
                conditions.append({"adaptive_support": filters["adaptive_support"]})
            if "max_duration" in filters and filters["max_duration"]:
                conditions.append({"duration": {"$lte": filters["max_duration"]}})
            if "test_types" in filters and filters["test_types"]:
                conditions.append({"test_types": {"$contains": filters["test_types"]}})
            
            if len(conditions) == 1:
                where_clause = conditions[0]
            elif len(conditions) > 1:
                where_clause = {"$and": conditions}
            print(f"⚙️ Applying DB Filters: {where_clause}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        query_args = {"query_embeddings": [query_embedding.tolist()], "n_results": top_k}
        if where_clause:
            query_args["where"] = where_clause

        results = self.vector_store.collection.query(**query_args)
        
        retrieved_docs = []
        if results['documents'] and results['documents'][0]:
            for i in range(len(results['documents'][0])):
                retrieved_docs.append({
                    'name': results['metadatas'][0][i]['name'],
                    'url': results['metadatas'][0][i]['url'],
                    'metadata': results['metadatas'][0][i],
                    'distance': results['distances'][0][i]
                })
        return retrieved_docs

# --- TEST COMPONENT 3 ---
retriever = AssessmentRetriever(vector_store, embedding_manager)

print("Test 1: Pure Semantic Search")
res1 = retriever.retrieve("AI developer")
print(f"Top result: {res1[0]['name'] if res1 else 'None'}\n")

print("Test 2: Semantic Search + Hard Filter")
res2 = retriever.retrieve("AI developer", filters={"remote_support": "Yes", "max_duration": 30})
print(f"Top result: {res2[0]['name'] if res2 else 'None'}")

Test 1: Pure Semantic Search

🔍 Retrieving top 5 for query: 'AI developer'
🧠 Generating embeddings for 1 texts...
Top result: AI Skills

Test 2: Semantic Search + Hard Filter

🔍 Retrieving top 5 for query: 'AI developer'
⚙️ Applying DB Filters: {'$and': [{'remote_support': 'Yes'}, {'duration': {'$lte': 30}}]}
🧠 Generating embeddings for 1 texts...
Top result: AI Skills


In [7]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Optional

# 1. Load the environment variables from your .env file
load_dotenv()

# Quick sanity check to ensure it loaded correctly
if not os.getenv("GROQ_API_KEY"):
    print("⚠️ WARNING: GROQ_API_KEY not found! Please check your .env file.")

# 2. Define the exact JSON schema we want using Pydantic
class AssessmentFilters(BaseModel):
    remote_support: Optional[str] = Field(default=None, description="'Yes' or 'No' only if explicitly requested.")
    adaptive_support: Optional[str] = Field(default=None, description="'Yes' or 'No' only if explicitly requested.")
    max_duration: Optional[str] = Field(default=None, description="Maximum test duration in minutes (e.g. '40').")

class QueryAnalysis(BaseModel):
    search_query: str = Field(description="The core semantic meaning of the job description or query.")
    filters: AssessmentFilters = Field(description="Explicit constraints mentioned in the query.")

class QueryRouter:
    """Analyzes raw text queries using an LLM to extract search parameters and filters."""
    def __init__(self, model_name: str = "llama-3.1-8b-instant"):
        print(f"🧠 Initializing LLM Router ({model_name})...")
        
        # ChatGroq will automatically look for os.environ["GROQ_API_KEY"] now!
        self.llm = ChatGroq(model=model_name, temperature=0)
        self.structured_llm = self.llm.with_structured_output(QueryAnalysis)
        
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an intelligent search router for an HR assessment recommendation engine. Analyze the user's job description or query and extract the core requirements."),
            ("human", "{query}")
        ])
        self.chain = self.prompt | self.structured_llm

    def analyze(self, query: str) -> QueryAnalysis:
        print(f"🤖 Routing query: '{query}'")
        return self.chain.invoke({"query": query})

# --- TEST COMPONENT 4 ---
router = QueryRouter()
analysis = router.analyze("I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.")
print("\n🎯 Extraction Results:")
print(analysis.model_dump_json(indent=4))

🧠 Initializing LLM Router (llama-3.1-8b-instant)...
🤖 Routing query: 'I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.'

🎯 Extraction Results:
{
    "search_query": "Java developer collaboration business teams",
    "filters": {
        "remote_support": null,
        "adaptive_support": null,
        "max_duration": "40"
    }
}


In [8]:
class AssessmentRecommender:
    """Orchestrates the LLM routing and Vector DB retrieval to generate final recommendations."""
    
    def __init__(self, router: QueryRouter, retriever: AssessmentRetriever):
        self.router = router
        self.retriever = retriever

    def _prepare_filters(self, llm_filters: AssessmentFilters) -> Dict[str, Any]:
        """Converts the Pydantic filter object into a dictionary for ChromaDB."""
        db_filters = {}
        if llm_filters.remote_support:
            db_filters["remote_support"] = llm_filters.remote_support
        if llm_filters.adaptive_support:
            db_filters["adaptive_support"] = llm_filters.adaptive_support
        if llm_filters.max_duration:
            # 🚨 FIX: Safely cast the string "40" back into the integer 40 for ChromaDB's math operator
            try:
                db_filters["max_duration"] = int(llm_filters.max_duration)
            except ValueError:
                pass # Ignore if the LLM hallucinates text instead of a number
                
        return db_filters

    def get_recommendations(self, raw_query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        print(f"\n🚀 Processing Request: '{raw_query}'")
        
        # 1. Analyze the query with the LLM
        analysis = self.router.analyze(raw_query)
        search_query = analysis.search_query
        base_filters = self._prepare_filters(analysis.filters)
        
        # 2. Execute single unified semantic search
        print("🎯 Executing unified semantic search...")
        final_results = self.retriever.retrieve(search_query, top_k=top_k, filters=base_filters)

        # 3. Format the final output to match assignment requirements
        formatted_output = []
        for res in final_results[:top_k]:
            formatted_output.append({
                "Assessment name": res["name"],
                "URL": res["url"],
                "Match Reason": f"Distance: {res['distance']:.4f} | Types: {res['metadata']['test_types']}",
                "Duration": f"{res['metadata']['duration']} mins",
                "Remote Support": res['metadata'].get('remote_support', 'N/A'),
                "Adaptive Support": res['metadata'].get('adaptive_support', 'N/A')
            })
            
        return formatted_output

# --- TEST COMPONENT 5 ---
# Initialize the orchestrator with the components from previous cells
recommender = AssessmentRecommender(router, retriever)

# The exact test scenario
test_query = "I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes."
recommendations = recommender.get_recommendations(test_query, top_k=5)

print("\n✅ FINAL RECOMMENDATIONS:")
print(json.dumps(recommendations, indent=4))


🚀 Processing Request: 'I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.'
🤖 Routing query: 'I am hiring for Java developers who can also collaborate effectively with my business teams. Looking for an assessment(s) that can be completed in 40 minutes.'
🎯 Executing unified semantic search...

🔍 Retrieving top 5 for query: 'Java developer collaboration business teams'
⚙️ Applying DB Filters: {'duration': {'$lte': 40}}
🧠 Generating embeddings for 1 texts...

✅ FINAL RECOMMENDATIONS:
[
    {
        "Assessment name": "Java 2 Platform Enterprise Edition 1.4 Fundamental",
        "URL": "https://www.shl.com/products/product-catalog/view/java-2-platform-enterprise-edition-1-4-fundamental/",
        "Match Reason": "Distance: 1.1868 | Types: Knowledge & Skills",
        "Duration": "30 mins",
        "Remote Support": "Yes",
        "Adaptive Support": "Yes"
    },
    {
        "Asses